In [ ]:
#Author: Paarth Sharma
#FileName: tinyStoriesGPT.ipynb
#ProjectName: TinyStoriesGPT
#CreationDate: 5-03-26
#ModificationDate: 19-03-26
#Description: This is a Transformer model trained on the tiny stories dataset using JAX and the XLA compiler for optimization

In [ ]:
#install grain for data loading
!pip install -q grain

In [ ]:
#importing libraries
from pathlib import Path
import jax
import flax.nnx as nnx
import jax.numpy as jnp
import optax
import orbax
import tiktoken
import grain.python as pygrain

maxlen = 128
num_epochs = 3
tokenizer = tiktoken.get_encoding("gpt2")
print(f"Vocabulary size: {tokenizer.n_vocab}")

In [ ]:
#mount google drive to access dataset and save checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#configure file paths
DRIVE_BASE = Path("/content/drive/MyDrive/Colab Notebooks/tinyStoriesProject")

file_path = DRIVE_BASE / "data/TinyStories-train.txt"

checkpoint_dir = DRIVE_BASE / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Part 1: Model Architecture

### Token + Positional Embedding

In [ ]:
class TokenEmbedding(nnx.Module):
  """
  Combines token embeddings and positional embeddings.
  Each token gets a learned vector representation, and each position
  in the sequence gets its own learned vector too. Both are added together.
  """

  #Pre : vocab_size, embedding_size, maxlen, rngs
  #Post : NA
  #Desc : sets up the two embedding tables - one for tokens, one for positions
  def __init__(self, vocab_size, embedding_size, maxlen, rngs):
    self.token_embedding = nnx.Embed(vocab_size, embedding_size, rngs=rngs)
    self.position_embedding = nnx.Embed(maxlen, embedding_size, rngs=rngs)

  #Pre : token_ids - integer array of shape (batch, seq_len)
  #Post : combined embedding of shape (batch, seq_len, embedding_size)
  #Desc : looks up token embeddings and adds positional information
  def __call__(self, token_ids):
    seq_len = token_ids.shape[1]
    positions = jnp.arange(seq_len).reshape(1, seq_len)
    return self.token_embedding(token_ids) + self.position_embedding(positions)

### Causal Attention Mask

In [ ]:
#Pre: sequence_length - int
#Post: lower triangular boolean mask of shape (1, 1, seq_len, seq_len)
#Desc: builds a mask that prevents each token from attending to future tokens.
#      this forces the model to only use past context when predicting the next token.
def causal_attention_mask(sequence_length):
  mask = jnp.tril(jnp.ones((sequence_length, sequence_length), dtype=bool))
  return mask[None, None, :, :]

### Transformer Block

In [ ]:
class TransformerBlock(nnx.Module):
  """
  Single transformer block with pre-layer normalization.
  Each block has two sub-components:
    1. Multi-head self attention (with causal mask)
    2. Feed forward network
  Both use residual connections and layer norm before the operation (Pre-LN).
  """

  #Pre : embedding_dimension, number_of_heads, feed_forward_dimension, rngs
  #Post : NA
  #Desc : initializes the attention, feed forward, and normalization layers
  def __init__(self, embedding_dimension, number_of_heads, feed_forward_dimension, *, rngs):

    self.layer_norm_1 = nnx.LayerNorm(embedding_dimension, rngs=rngs)

    self.attention = nnx.MultiHeadAttention(
        in_features=embedding_dimension,
        qkv_features=embedding_dimension,
        out_features=embedding_dimension,
        num_heads=number_of_heads,
        rngs=rngs,
        decode=False
    )

    self.layer_norm_2 = nnx.LayerNorm(embedding_dimension, rngs=rngs)

    self.feed_forward_1 = nnx.Linear(embedding_dimension, feed_forward_dimension, rngs=rngs)
    self.feed_forward_2 = nnx.Linear(feed_forward_dimension, embedding_dimension, rngs=rngs)

  #Pre : tokens of shape (batch, seq_len, embedding_dim), optional mask
  #Post : transformed tokens of same shape
  #Desc : runs attention then feed forward, both with pre-norm and residual connections
  def __call__(self, tokens, mask=None):

    # attention sub-block with pre-norm and residual
    normalized = self.layer_norm_1(tokens)
    attention_out = self.attention(normalized, mask=mask)
    tokens = tokens + attention_out

    # feed forward sub-block with pre-norm and residual
    normalized_2 = self.layer_norm_2(tokens)
    hidden = self.feed_forward_1(normalized_2)
    hidden = nnx.gelu(hidden)
    hidden = self.feed_forward_2(hidden)
    tokens = tokens + hidden

    return tokens

### Full GPT Model

In [ ]:
class miniGPTModel(nnx.Module):
  """
  A small GPT-style language model.
  Architecture:
    - Token + positional embedding layer
    - N stacked transformer blocks with causal masking
    - Final layer norm
    - Linear projection to vocabulary size (logits)
  """

  #Pre: maxlen, vocab_size, embedding_dimension, number_of_heads,
  #     feed_forward_dimension, number_of_transformer_blocks, rngs
  #Post: NA
  #Desc: builds all layers of the model
  def __init__(self, maxlen, vocab_size, embedding_dimension, number_of_heads,
               feed_forward_dimension, number_of_transformer_blocks, *, rngs):

    self.maxlen = maxlen
    self.embedding = TokenEmbedding(vocab_size, embedding_dimension, maxlen, rngs=rngs)

    self.transformer_blocks = []
    for _ in range(number_of_transformer_blocks):
      block = TransformerBlock(embedding_dimension, number_of_heads, feed_forward_dimension, rngs=rngs)
      self.transformer_blocks.append(block)

    self.final_layer_norm = nnx.LayerNorm(embedding_dimension, rngs=rngs)
    self.output_layer = nnx.Linear(embedding_dimension, vocab_size, use_bias=False, rngs=rngs)

  #Pre: token_ids - integer array of shape (batch, seq_len)
  #Post: logits of shape (batch, seq_len, vocab_size)
  #Desc: full forward pass through embedding -> transformer blocks -> output projection
  def __call__(self, token_ids):
    seq_len = token_ids.shape[1]
    mask = causal_attention_mask(seq_len)
    x = self.embedding(token_ids)

    for block in self.transformer_blocks:
      x = block(x, mask)

    x = self.final_layer_norm(x)
    return self.output_layer(x)

In [ ]:
#instantiate the model
model = miniGPTModel(
    maxlen=128,
    vocab_size=tokenizer.n_vocab,
    embedding_dimension=256,
    number_of_heads=8,
    feed_forward_dimension=1024,
    number_of_transformer_blocks=6,
    rngs=nnx.Rngs(0)
)

# print parameter count
n_params = sum(x.size for x in jax.tree.leaves(nnx.state(model)))
print(f"Total parameters: {n_params:,}")

# Part 2: Data Loading

### Dataset Class

In [ ]:
class Dataset:

  #pre : stories(list of strings), maxlen(int), tokenizer
  #post: NA
  #desc: stores stories and tokenizer, records the end token id
  def __init__(self, stories, maxlen, tokenizer):
    self.stories = stories
    self.maxlen = maxlen
    self.tokenizer = tokenizer
    self.endtoken = tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"})[0]

  #pre : NA
  #post: number of stories in the dataset
  #desc: required by grain to know how many records exist
  def __len__(self):
    return len(self.stories)

  #pre : index(int)
  #post: fixed-length token array of size maxlen
  #desc: tokenizes a story, truncates if too long, pads with zeros if too short
  def __getitem__(self, index):
    story = self.stories[index]
    tokens = self.tokenizer.encode(story, allowed_special={"<|endoftext|>"})

    if len(tokens) > self.maxlen:
      tokens = tokens[:self.maxlen]

    tokens += [0] * (self.maxlen - len(tokens))
    return tokens

### Data Loading Function

In [ ]:
#pre : file_path, batch_size, maxlen, max_stories, num_epochs, shuffle, seed
#post: dataloader, estimated_batches_per_epoch
#desc: reads the raw TinyStories text file, splits it into individual stories
#      using the <|endoftext|> token as a delimiter, then wraps them in a
#      grain DataLoader for efficient batched training
def load_and_process_dataset(
    file_path,
    batch_size,
    maxlen,
    max_stories,
    num_epochs,
    shuffle,
    seed=60
):
  stories = []
  current_story = []

  with open(file_path, "r", encoding='utf-8') as file:
    for line in file:

      if "<|endoftext|>" in line:
        parts = line.split('<|endoftext|>')

        for part in parts[:-1]:
          current_story.append(part)
          story_text = "".join(current_story).strip()

          if story_text:
            stories.append(story_text + '<|endoftext|>')
            if len(stories) >= max_stories:
              break
          current_story = []

        if parts[-1].strip():
          current_story = [parts[-1]]

        if len(stories) >= max_stories:
          break
      else:
        current_story.append(line)

  estimated_batches_per_epoch = len(stories) // batch_size
  print(f"Loaded {len(stories)} stories | Steps per epoch: {estimated_batches_per_epoch}")

  dataset = Dataset(stories, maxlen, tokenizer)

  sampler = pygrain.IndexSampler(
    num_records=len(dataset),
    shuffle=shuffle,
    seed=seed,
    shard_options=pygrain.NoSharding(),
    num_epochs=1,  
  )

  dataloader = pygrain.DataLoader(
    data_source=dataset,
    sampler=sampler,
    operations=[
      pygrain.Batch(batch_size=batch_size, drop_remainder=True)
    ]
  )

  return dataloader, estimated_batches_per_epoch

In [ ]:
data_loader, estimated_batches_per_epoch = load_and_process_dataset(
    file_path=file_path,
    batch_size=32,
    maxlen=maxlen,
    max_stories=100000,
    num_epochs=num_epochs,
    shuffle=True,
    seed=60
)

# Part 3: Training

### Loss Function

In [ ]:
#pre : model, batch tuple of (inputs, targets)
#post: (scalar loss, logits)
#desc: runs a forward pass and computes cross entropy loss between
#      the model predictions and the target token ids
def compute_loss(model, batch):
  inputs, targets = batch
  predictions = model(inputs)

  loss = optax.softmax_cross_entropy_with_integer_labels(
      predictions, targets
  ).mean()

  return loss, predictions

### Learning Rate Schedule

In [ ]:
total_steps = estimated_batches_per_epoch * num_epochs
warmup_steps = max(1, total_steps // 10)

# warmup for first 10% of steps, then cosine decay to near zero
lr_schedule = optax.warmup_cosine_decay_schedule(
    init_value=0.0,
    peak_value=3e-4,
    warmup_steps=warmup_steps,
    decay_steps=total_steps,
    end_value=1e-5
)

print(f"Total training steps: {total_steps} | Warmup steps: {warmup_steps}")

### Metrics and Optimizer

In [ ]:
metrics = nnx.MultiMetric(
    loss=nnx.metrics.Average('loss'),
)

optimizer = nnx.Optimizer(
    model,
    optax.adamw(learning_rate=lr_schedule, weight_decay=0.01),
    wrt=nnx.Param
)

### Training Step

In [ ]:
#pre : model, optimizer, metrics, batch
#post: NA (updates model weights and metrics in place)
#desc: computes gradients of the loss with respect to model parameters,
#      updates weights via the optimizer, and logs the loss to metrics
@nnx.jit
def train_step(model, optimizer, metrics, batch):
    (loss, predictions), gradients = nnx.value_and_grad(
        compute_loss, has_aux=True
    )(model, batch)

    optimizer.update(model, gradients)
    metrics.update(loss=loss)

### Training Loop

In [ ]:
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
metrics_history = {'train_loss': []}
checkpoint_every_n_steps = 100

# shift tokens left by 1 to create targets — each token predicts the next
def create_targets(batch):
  targets = jnp.concatenate(
      [batch[:, 1:], jnp.zeros((batch.shape[0], 1), dtype=jnp.int32)],
      axis=1
  )
  return targets

for epoch in range(num_epochs):
    step = 0

    for batch in data_loader:
        input_batch = jnp.array(jnp.array(batch).T).astype(jnp.int32)
        target_batch = create_targets(input_batch)

        print(".", end="")

        train_step(model, optimizer, metrics, (input_batch, target_batch))

        if (step + 1) % 2 == 0:
            for metric, value in metrics.compute().items():
                metrics_history[f'train_{metric}'].append(value)
            metrics.reset()

            current_lr = lr_schedule(step)
            print(f"\nEpoch: {epoch + 1}, Step: {step + 1}, "
                  f"Loss: {metrics_history['train_loss'][-1]:.4f}, "
                  f"LR: {current_lr:.2e}")

        if (step + 1) % checkpoint_every_n_steps == 0:
            ckpt_path = checkpoint_dir / f"epoch_{epoch+1}_step_{step+1}"
            checkpointer.save(ckpt_path, nnx.state(model), force=True)
            print(f"\nCheckpoint saved: {ckpt_path}")

        step += 1

    epoch_ckpt_path = checkpoint_dir / f"epoch_{epoch+1}_final"
    checkpointer.save(epoch_ckpt_path, nnx.state(model), force=True)
    print(f"\nEpoch {epoch+1} complete. Checkpoint saved: {epoch_ckpt_path}")

# Part 4: Inference

### Load Checkpoint

In [ ]:
import os

# list available checkpoints
print("Available checkpoints:")
print(sorted(os.listdir(checkpoint_dir)))

In [ ]:
checkpointer = orbax.checkpoint.PyTreeCheckpointer()
restored_state = checkpointer.restore(checkpoint_dir / "epoch_1_final")

graphdef, _ = nnx.split(model)
model = nnx.merge(graphdef, restored_state)
print("Checkpoint loaded successfully")

### Generation Function

In [ ]:
#pre : model, prompt(str), max_new_tokens(int), temperature(float)
#post: generated story string
#desc: autoregressively generates tokens one at a time using temperature sampling.
def generate(model, prompt, max_new_tokens=200, temperature=0.5):
    model.eval()
    tokens = tokenizer.encode(prompt)
    rng = jax.random.PRNGKey(42)
    end_token = tokenizer.encode("<|endoftext|>",
                    allowed_special={"<|endoftext|>"})[0]

    for _ in range(max_new_tokens):
        # pad to fixed maxlen to avoid JAX recompilation each step
        padded = tokens[-maxlen:]
        pad_len = maxlen - len(padded)
        input_ids = jnp.array([0] * pad_len + padded).reshape(1, -1)

        logits = model(input_ids)[0, -1, :]

        # temperature sampling
        probs = jax.nn.softmax(logits / temperature)
        rng, subkey = jax.random.split(rng)
        next_token = int(jax.random.choice(subkey, len(probs), p=probs))

        tokens.append(next_token)

        if next_token == end_token:
            break

    return tokenizer.decode(tokens).replace("\\n", "\n")

In [ ]:
prompts = [
    "Once upon a time in a forest",
    "There was a little dog named",
    "The sun was shining and",
]

for prompt in prompts:
    print(f"Prompt: {prompt}")
    print(generate(model, prompt, max_new_tokens=200, temperature=0.5))
    print("-" * 60)